# ***Design & Build a RAG-Based Customer Support Assistant***

## **Install Required Libraries**

In [93]:
!pip install -U langchain-community langchain-text-splitters langchain-huggingface langchain-openai langchain-groq chromadb pypdf langgraph

## **Import Dependencies**

In [94]:
import os

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

from langchain_groq import ChatGroq

from langgraph.graph import StateGraph

In [95]:
import shutil
shutil.rmtree("chroma_db", ignore_errors=True)

## **Upload PDF**

In [96]:
from google.colab import files
uploaded = files.upload()
PDF_PATH = "Forward Learning Workbook (Updated).pdf"
loader = PyPDFLoader(PDF_PATH)
docs = loader.load()

Saving Forward Learning Workbook (Updated).pdf to Forward Learning Workbook (Updated).pdf


## **Preview Loaded Data**

In [97]:
print(docs[0].page_content[:333])



CONFIDENTIAL AND PROPRIETARY | © 2025 McKinsey & Company. 
This material is intended solely for your internal use and any use of this material without 
specific permission of McKinsey & Company is strictly prohibited. All rights reserved.
Take a Step.
Forward.
My Learning Workbook
Your Name: ________________________


In [99]:
loader = PyPDFLoader("Forward Learning Workbook (Updated).pdf")
documents = loader.load()

print(f"Loaded {len(documents)} pages")

Loaded 65 pages


## **Split Text into Chunks**

In [100]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)
print(f"Total chunks: {len(chunks)}")

Total chunks: 231


## **Create Embeddings & Vector Database**

In [101]:
import os
import shutil
import chromadb
from chromadb.config import Settings

# Explicitly ensure the persistence directory is clean just before Chroma tries to use it
if os.path.exists("./chroma_db"):
    shutil.rmtree("./chroma_db", ignore_errors=True) # Remove to clear any locks/corruption
# Ensure the directory exists (Chroma might create it, but being explicit doesn't hurt)
os.makedirs("./chroma_db", exist_ok=True)

embedding = HuggingFaceEmbeddings()

# Explicitly initialize the Chroma client for persistence
client = chromadb.Client(Settings(persist_directory="./chroma_db"))

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding,
    client=client, # Pass the configured client
    collection_name="my_rag_collection" # It's good practice to name the collection
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## **LLM SETUP (GROQ API)**

In [102]:
os.environ["GROQ_API_KEY"] = "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"

llm = ChatGroq(
    model_name="llama-3.1-8b-instant"
)

## **RETRIEVER SETUP**

In [103]:
def retrieve_docs(query):
  docs = retriever.invoke(query)
  return docs

## **Generate Answer Using RAG**

In [104]:
def generate_answer(query, docs):

  if not docs:
    return " This question is outside the document scope. Redirecting to human support."

  context = "\n\n".join([doc.page_content for doc in docs])

  prompt = f"""

You are a customer support assistant.docs

Answer the question based only on the context below.
If the question cannot be answered using the information
provided answer with "I don't know".

Be polite and helpful.

Context:
{context}

Question:
{query}

Answer:
"""

  response = llm.invoke(prompt)
  return response.content

## **Define Graph State**

In [105]:
from typing import TypedDict, List
from langchain_core.documents import Document

class GraphState(TypedDict):
  query: str
  docs: List[Document]
  answer: str

## **Processing Node**

In [106]:
def process_node(state):
  query = state.get("query", "")
  docs = retrieve_docs(query)

  return {
      "query": query,
      "docs": docs
  }

## **Output Node**

In [107]:
def answer_node(state):
  query = state.get("query", "")
  docs = state.get("docs", [])

  answer = generate_answer(query, docs)

  return {
      "query": query,
      "docs": docs,
      "answer": answer
  }

## **HITL Node**

In [108]:
def hitl_node(state):
  return {
      "answer": " This query is escalated to human support."
  }

## **Routing Logic**

In [109]:
def route(state):
  docs=state.get("docs",[])

  if len(docs) == 0:
    return "hitl"

  if len(docs) < 2:
    return "hitl"

  return "answer"

## **Build Graph**

In [110]:
graph = StateGraph(GraphState)

graph.add_node("process", process_node)
graph.add_node("answer", answer_node)
graph.add_node("hitl", hitl_node)

graph.set_entry_point("process")

graph.add_conditional_edges(
    "process",
    route,
    {
        "answer": "answer",
        "hitl": "hitl"
    }
)

graph.set_finish_point("answer")
graph.set_finish_point("hitl")

app = graph.compile()

## **Run Query**

In [111]:
while True:
  query = input("\nAsk your question (type 'exit' to stop): ")

  if query.lower() == "exit":
    print("Chat ended.")
    break

  result = app.invoke({
      "query": query,
      "docs": [],
      "answer": ""
  })

  print("\n Answer:")
  print(result["answer"])


Ask your question (type 'exit' to stop): what's the program about?

 Answer:
The program appears to be focused on personal and professional development, specifically on learning effective learning habits, mindsets for change, leadership skills, digital elements, and problem-solving techniques. It seems to be a comprehensive program aimed at enhancing future of work skills and behaviors.

Ask your question (type 'exit' to stop): what are the advantages of this program?

 Answer:
Based on the information provided, the advantages of this program seem to be:

* Gaining effective learning habits and mindsets for change through 4 self-paced modules
* Applying and deepening leadership skills
* Receiving a McKinsey.org Forward digital badge upon completion
* Joining the exclusive global network of lifelong learners, Network Level
* Having a tool, YourLearning Workbook, to capture learnings, ideas, and reflections during and after the program
* Gaining insights into strengths and areas of impr